# Model Input Analsysis 

This notebook examines the train, test and validation examples, analyzes persona and sequence lengths.

## Prerequisite

Generate dataset split with compact prompts
```bash
uv run python scripts/preprocess_data.py \
  --data-config configs/data/twin2k500.yaml
```


In [ ]:
from datasets import load_from_disk
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import numpy as np
import pandas as pd

In [ ]:
participant_splits = load_from_disk(
    "../data/processed/twin2k500_compact"
)

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    use_fast=True,
)

In [ ]:
prompt_splits = load_from_disk(
    PROMPT_DATA_DIR
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
)

In [ ]:
split_names = [
    "train",
    "validation",
    "test",
]

total_participants = sum(
    len(set(participant_splits[name]["pid"]))
    for name in split_names
)

records = []

for split_name in split_names:
    participant_count = len(
        set(participant_splits[split_name]["pid"])
    )
    example_count = len(
        prompt_splits[split_name]
    )

    records.append(
        {
            "split": split_name.title(),
            "participants": participant_count,
            "examples": example_count,
            "percentage": (
                100
                * participant_count
                / total_participants
            ),
        }
    )

split_summary = pd.DataFrame(records)

split_summary

## Persona compression

In [ ]:
def persona_token_lengths(
    texts,
    *,
    batch_size: int = 16,
    description: str = "Tokenizing personas",
) -> np.ndarray:
    lengths = []

    with tqdm(
        total=len(texts),
        desc=description,
        unit="persona",
    ) as progress:
        for start in range(
            0,
            len(texts),
            batch_size,
        ):
            batch = texts[
                start : start + batch_size
            ]

            encoded = tokenizer(
                batch,
                add_special_tokens=False,
                return_length=True,
                truncation=False,
            )

            lengths.extend(encoded["length"])
            progress.update(len(batch))

    return np.asarray(
        lengths,
        dtype=np.int64,
    )

In [ ]:
records = []

for split_name in [
    "train",
    "validation",
    "test",
]:
    split = participant_splits[split_name]

    raw_lengths = persona_token_lengths(
        split["wave1_3_persona_text"],
        description=f"{split_name}: raw personas",
    )

    compact_lengths = persona_token_lengths(
        split["wave1_3_compact_persona_text"],
        description=f"{split_name}: compact personas",
    )

    records.append(
        {
            "split": split_name,
            "participants": len(split),
            "raw_tokens_mean": raw_lengths.mean(),
            "raw_tokens_median": np.median(
                raw_lengths
            ),
            "raw_tokens_max": raw_lengths.max(),
            "compact_tokens_mean": (
                compact_lengths.mean()
            ),
            "compact_tokens_median": np.median(
                compact_lengths
            ),
            "compact_tokens_max": (
                compact_lengths.max()
            ),
            "mean_reduction_pct": (
                100
                * (
                    1
                    - compact_lengths
                    / raw_lengths
                ).mean()
            ),
        }
    )

persona_summary = (
    pd.DataFrame(records)
    .set_index("split")
    .round(1)
)

persona_summary